# "THE POUR IS RIGHT" - Wine Pricer Capstone

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we write some simple models to predict the price of a wine, and a way to score them. This follows `week6/day3.ipynb`.

This is also the go/no-go test for the whole project. If a model that reads the words clearly beats one that ignores them, the text carries a signal and everything in weeks 7 and 8 is worth doing.

Everything today runs on your own machine and costs nothing.

In [ ]:
import random
import pandas as pd
import numpy as np
from huggingface_hub import HfApi
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

load_dotenv(override=True)

In [ ]:
username = HfApi().whoami()["name"]
dataset = f"{username}/wines_lite"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## How we score a model

`evaluate()` is Ed's, from `pricer/evaluator.py`. You give it a function that takes a wine and returns a price. It runs that function on the first 200 test wines and reports:

- the **average error** in dollars: how far off the guesses are, on average. This is the number we compare between models.
- a chart of that average as more wines are scored, with a grey band showing how sure we can be of it
- a scatter plot of guess against actual price. A perfect model would put every dot on the dashed line.

Each dot is coloured by how good the guess was. Ed's limits were changed, because his were set for products up to \$999 and our wines stop at \$200:

- green: within \$10, or within 20% of the real price
- orange: within \$20, or within 40%
- red: anything worse

Every model, today and for the rest of the project, is scored on the same 200 wines, so the numbers can be compared fairly.

## Model 1: a random guess

The worst sensible model: pick any price between \$4 and \$200. Expect an error of roughly \$75.

In [ ]:
def random_pricer(item):
    return random.randrange(4, 200)

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

## Model 2: always guess the average

This is the baseline that matters. It knows nothing about the wine in front of it; it just answers with the average training price every time. Any real model has to beat it. Expect about \$21.70.

In [ ]:
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

## Model 3: linear regression on two numbers

Ed used the product's weight and the length of its text. Wine has no weight, so we use the reviewer's points and the length of the summary.

Linear regression finds the straight-line rule that fits best: price = a x points + b x length + c. Expect about \$16.50.

In [ ]:
def get_features(item):
    return {
        "points": item.points,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# Traditional Linear Regression!

np.random.seed(42)

feature_columns = ['points', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

The `points` number above is how many dollars the model adds for each extra point. It should be about 5: one more point from the reviewer is worth roughly \$5 on the price.

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
evaluate(linear_regression_pricer, test)

## Model 4: let the model read the words

Now the first model that looks at the text. A "bag of words" counts how often each word appears in a summary, ignoring word order. We keep the 2,000 most common words, so each wine becomes a list of 2,000 counts, and linear regression learns how many dollars each word is worth.

Expect about \$13.50. This is the go/no-go result.

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

In [ ]:
# Here are some of the 2,000 words that it picked, not including "stop words" like "the" and "and"

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
# Which words push the price up the most, and which pull it down?

word_values = sorted(zip(regressor.coef_, selected_words))
print("Words that lower the price:", [word for value, word in word_values[:15]])
print("Words that raise the price:", [word for value, word in word_values[-15:]])

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

## Model 5: Random Forest

A decision tree predicts by asking a chain of yes/no questions about the features, like a flow chart: does "Burgundy" appear? Is "90" there? A single tree is quick but tends to memorise its training data.

A Random Forest builds 100 trees, each on a different random part of the data and a different random set of words, then averages their answers.

It is slow. On this laptop the full 20,000 wines took 6.5 minutes, so, like Ed, we train on a subset. With 10,000 wines it takes about 2 minutes. Expect about \$12.

In [ ]:
subset = 10_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

## Model 6: XGBoost

XGBoost also combines many decision trees, but builds them one after another, each new tree correcting the mistakes of the ones before it.

It is far faster than Random Forest (about 10 seconds for all 20,000 wines) and usually a little more accurate. Expect about \$11.70.

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)

## Two questions from days 1 and 2

Now that we can score models, we can answer two things we left open. Both use the bag-of-words linear regression (model 4, about \$13.50) as the reference.

### How much do the models lean on the points score?

A wine on a shop page often has no critic's score. Here we remove the `Points:` line from every summary and train again. Expect about \$14.70, so the score is worth a little over \$1 of accuracy. Useful, but the words do most of the work.

In [ ]:
def without_points(item):
    return "\n".join(line for line in item.summary.split("\n") if not line.startswith("Points:"))

vectorizer_np = CountVectorizer(max_features=2000, stop_words='english')
X_np = vectorizer_np.fit_transform([without_points(item) for item in train])
regressor_np = LinearRegression().fit(X_np, prices)

def nlp_lr_without_points(item):
    x = vectorizer_np.transform([without_points(item)])
    return max(regressor_np.predict(x)[0], 0)

evaluate(nlp_lr_without_points, test)

### Did the rewrite on day 2 lose information?

Here we train on the original `full` text instead of the summary. Expect about \$13.10, against \$13.50 for the summaries.

So the rewrite cost about 40 cents of accuracy. Part of that gap is the price clues we removed on purpose: the original text still has all 693 of them.

In [ ]:
vectorizer_full = CountVectorizer(max_features=2000, stop_words='english')
X_full = vectorizer_full.fit_transform([item.full for item in train])
regressor_full = LinearRegression().fit(X_full, prices)

def nlp_lr_full_text(item):
    x = vectorizer_full.transform([item.full])
    return max(regressor_full.predict(x)[0], 0)

evaluate(nlp_lr_full_text, test)

## Results so far

Replace the numbers below with the errors from your own runs, then run the cell to see the chart. The values filled in are what I got when testing.

In [ ]:
import plotly.graph_objects as go

results = [
    ("Constant", "gray", 21.71),
    ("Linear Regression", "gray", 16.46),
    ("NLP + LR", "gray", 13.50),
    ("Random Forest", "gray", 12.01),
    ("XGBoost", "gray", 11.67),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, text=[f"${v:.2f}" for v in values], textposition="outside"))

fig.update_layout(
    title="Wine pricer - average error by model (lower is better)",
    yaxis=dict(range=[0, max(values) * 1.15], title="Error ($)"),
    width=800,
    height=600
)

fig.show()